<a href="https://colab.research.google.com/github/dharshini-dev-hub/bigdata-lab/blob/rdd-operations/Multiple_sheets_pyspark_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
!wget https://repo1.maven.org/maven2/com/crealytics/spark-excel_2.12/3.5.0_0.20.3/spark-excel_2.12-3.5.0_0.20.3.jar -O /content/spark-excel.jar

--2025-10-18 04:35:46--  https://repo1.maven.org/maven2/com/crealytics/spark-excel_2.12/3.5.0_0.20.3/spark-excel_2.12-3.5.0_0.20.3.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:120c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30930051 (29M) [application/java-archive]
Saving to: ‘/content/spark-excel.jar’

/content/spark-exce 100%[===================>]  29.50M   108MB/s    in 0.3s    

2025-10-18 04:35:46 (108 MB/s) - ‘/content/spark-excel.jar’ saved [30930051/30930051]



In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ExcelExample") \
    .config("spark.jars", "/content/spark-excel.jar") \
    .getOrCreate()

In [4]:
file_path = "/content/data.xlsx.xlsx"

customers_df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("dataAddress", "'Customers'!A1") \
    .load(file_path)

customers_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|        1.0|        Alice|  Chennai|
|        2.0|          Bob|Bangalore|
|        3.0|      Charlie|    Delhi|
+-----------+-------------+---------+



In [5]:
products_df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("dataAddress", "'Products'!A1") \
    .load(file_path)

products_df.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|     101.0|     Biscuit| 20.0|
|     102.0|       Juice| 40.0|
|     103.0|       Chips| 30.0|
+----------+------------+-----+



In [6]:
orders_df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("dataAddress", "'Orders'!A1") \
    .load(file_path)

orders_df.show()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|  1001.0|        1.0|     101.0|     2.0|
|  1002.0|        2.0|     102.0|     1.0|
|  1003.0|        1.0|     103.0|     3.0|
|  1004.0|        3.0|     101.0|     5.0|
+--------+-----------+----------+--------+



In [7]:
customers_df.printSchema()
products_df.printSchema()
orders_df.printSchema()

root
 |-- customer_id: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- product_id: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)

root
 |-- order_id: double (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- product_id: double (nullable = true)
 |-- quantity: double (nullable = true)



In [8]:
#Add a column

new_row = [(1005.0, 4.0, 102.0, 2.0)]
new_row_df = spark.createDataFrame(new_row, orders_df.columns)
orders_df = orders_df.union(new_row_df)
orders_df.show()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|  1001.0|        1.0|     101.0|     2.0|
|  1002.0|        2.0|     102.0|     1.0|
|  1003.0|        1.0|     103.0|     3.0|
|  1004.0|        3.0|     101.0|     5.0|
|  1005.0|        4.0|     102.0|     2.0|
+--------+-----------+----------+--------+



In [9]:
# Inner Join

order_customer_df = orders_df.join(customers_df, "customer_id", "inner")
order_customer_df.show()

+-----------+--------+----------+--------+-------------+---------+
|customer_id|order_id|product_id|quantity|customer_name|     city|
+-----------+--------+----------+--------+-------------+---------+
|        1.0|  1001.0|     101.0|     2.0|        Alice|  Chennai|
|        1.0|  1003.0|     103.0|     3.0|        Alice|  Chennai|
|        2.0|  1002.0|     102.0|     1.0|          Bob|Bangalore|
|        3.0|  1004.0|     101.0|     5.0|      Charlie|    Delhi|
+-----------+--------+----------+--------+-------------+---------+



In [10]:
# Left join

left_join_df = customers_df.join(orders_df, "customer_id", "left")
left_join_df.show()

+-----------+-------------+---------+--------+----------+--------+
|customer_id|customer_name|     city|order_id|product_id|quantity|
+-----------+-------------+---------+--------+----------+--------+
|        1.0|        Alice|  Chennai|  1003.0|     103.0|     3.0|
|        1.0|        Alice|  Chennai|  1001.0|     101.0|     2.0|
|        3.0|      Charlie|    Delhi|  1004.0|     101.0|     5.0|
|        2.0|          Bob|Bangalore|  1002.0|     102.0|     1.0|
+-----------+-------------+---------+--------+----------+--------+



In [11]:
# Right join

right_join_df = customers_df.join(orders_df, "customer_id", "right")
right_join_df.show()

+-----------+-------------+---------+--------+----------+--------+
|customer_id|customer_name|     city|order_id|product_id|quantity|
+-----------+-------------+---------+--------+----------+--------+
|        1.0|        Alice|  Chennai|  1001.0|     101.0|     2.0|
|        2.0|          Bob|Bangalore|  1002.0|     102.0|     1.0|
|        1.0|        Alice|  Chennai|  1003.0|     103.0|     3.0|
|        3.0|      Charlie|    Delhi|  1004.0|     101.0|     5.0|
|        4.0|         NULL|     NULL|  1005.0|     102.0|     2.0|
+-----------+-------------+---------+--------+----------+--------+



In [14]:
# Full outer join

full_join_df = customers_df.join(orders_df, "customer_id", "outer")
full_join_df.show()

+-----------+-------------+---------+--------+----------+--------+
|customer_id|customer_name|     city|order_id|product_id|quantity|
+-----------+-------------+---------+--------+----------+--------+
|        1.0|        Alice|  Chennai|  1001.0|     101.0|     2.0|
|        1.0|        Alice|  Chennai|  1003.0|     103.0|     3.0|
|        2.0|          Bob|Bangalore|  1002.0|     102.0|     1.0|
|        3.0|      Charlie|    Delhi|  1004.0|     101.0|     5.0|
|        4.0|         NULL|     NULL|  1005.0|     102.0|     2.0|
+-----------+-------------+---------+--------+----------+--------+



In [15]:
# Outer join

full_data = orders_df.join(customers_df, "customer_id", "inner") \
                     .join(products_df, "product_id", "inner")
full_data.show()

+----------+-----------+--------+--------+-------------+---------+------------+-----+
|product_id|customer_id|order_id|quantity|customer_name|     city|product_name|price|
+----------+-----------+--------+--------+-------------+---------+------------+-----+
|     101.0|        3.0|  1004.0|     5.0|      Charlie|    Delhi|     Biscuit| 20.0|
|     101.0|        1.0|  1001.0|     2.0|        Alice|  Chennai|     Biscuit| 20.0|
|     103.0|        1.0|  1003.0|     3.0|        Alice|  Chennai|       Chips| 30.0|
|     102.0|        2.0|  1002.0|     1.0|          Bob|Bangalore|       Juice| 40.0|
+----------+-----------+--------+--------+-------------+---------+------------+-----+



In [16]:
# Total qty ordered per customer

from pyspark.sql.functions import *

total_quantity_per_customer = order_customer_df.groupBy("customer_name").agg(sum("quantity").alias("total_quantity"))
total_quantity_per_customer.show()

+-------------+--------------+
|customer_name|total_quantity|
+-------------+--------------+
|      Charlie|           5.0|
|          Bob|           1.0|
|        Alice|           5.0|
+-------------+--------------+



In [17]:
# Total sales per customer
from pyspark.sql.functions import sum
from pyspark.sql import functions as F

total_sales_per_customer = full_data.withColumn("total_price", F.col("quantity") * F.col("price"))\
                                    .groupBy("customer_name")\
                                    .agg(sum("total_price").alias("total_sales"))
total_sales_per_customer.show()

+-------------+-----------+
|customer_name|total_sales|
+-------------+-----------+
|      Charlie|      100.0|
|          Bob|       40.0|
|        Alice|      130.0|
+-------------+-----------+



In [18]:
# Total orders per city

total_orders_per_city = order_customer_df.groupBy("city")\
                                         .agg(count("order_id").alias("total_orders"))\
                                         .orderBy("total_orders", ascending=False)\

total_orders_per_city.show()

+---------+------------+
|     city|total_orders|
+---------+------------+
|  Chennai|           2|
|Bangalore|           1|
|    Delhi|           1|
+---------+------------+



In [19]:
# Total sales per product

total_sales_per_product = full_data.withColumn("total_price", F.col("quantity") * F.col("price"))\
                                   .groupBy("product_name")\
                                   .agg(sum("total_price").alias("total_sales"))\
                                   .orderBy(F.desc("total_sales"))
total_sales_per_product.show()

+------------+-----------+
|product_name|total_sales|
+------------+-----------+
|     Biscuit|      140.0|
|       Chips|       90.0|
|       Juice|       40.0|
+------------+-----------+



In [23]:
# Orders count per customer per product

orders_count = full_data.groupBy("customer_name")\
                        .pivot("product_name")\
                        .agg(F.sum("quantity"))\
                        .fillna(0)\

orders_count.show()

+-------------+-------+-----+-----+
|customer_name|Biscuit|Chips|Juice|
+-------------+-------+-----+-----+
|      Charlie|    5.0|  0.0|  0.0|
|          Bob|    0.0|  0.0|  1.0|
|        Alice|    2.0|  3.0|  0.0|
+-------------+-------+-----+-----+



In [22]:
# Total revenue per city

total_revenue_per_city = full_data.withColumn("total_price", F.col("quantity")*F.col("price"))\
                                  .groupBy("city")\
                                  .agg(sum("total_price").alias("total_revenue"))\
                                  .orderBy(F.desc("total_revenue"))

total_revenue_per_city.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|  Chennai|        130.0|
|    Delhi|        100.0|
|Bangalore|         40.0|
+---------+-------------+



In [25]:
# Total revenue per customer per product category

revenue_per_customer_product = full_data.withColumn("total_price", F.col("quantity") * F.col("price")) \
    .groupBy("customer_name", "product_name") \
    .agg(F.sum("total_price").alias("total_spent")) \
    .orderBy("customer_name", "product_name")

revenue_per_customer_product.show()

+-------------+------------+-----------+
|customer_name|product_name|total_spent|
+-------------+------------+-----------+
|        Alice|     Biscuit|       40.0|
|        Alice|       Chips|       90.0|
|          Bob|       Juice|       40.0|
|      Charlie|     Biscuit|      100.0|
+-------------+------------+-----------+



In [27]:
# Top N customers by total spend

top_spent_per_customer = full_data.withColumn("total_price", F.col("quantity")*F.col("price"))\
                                  .groupBy("customer_name")\
                                  .agg(sum("total_price").alias("total_spent"))\
                                  .orderBy(F.desc("total_spent"))\
                                  .limit(2)
top_spent_per_customer.show()

+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Alice|      130.0|
|      Charlie|      100.0|
+-------------+-----------+



In [29]:
# Most Popular Product by Quantity Sold

popular_product = products_df.join(orders_df, "product_id", "inner")\
                             .groupBy("product_name")\
                             .agg(F.sum("quantity").alias("total_quantity"))\
                             .orderBy(F.desc("total_quantity"))\
                             .limit(1)
popular_product.show()

+------------+--------------+
|product_name|total_quantity|
+------------+--------------+
|     Biscuit|           7.0|
+------------+--------------+



In [32]:
# Average Order Value per Customer

avg_order_value = full_data.withColumn("order_total", F.col("quantity")*F.col("price"))\
                           .groupBy("customer_name","order_id")\
                           .agg(F.sum("order_total").alias("total_order_value"))\
                           .groupBy("customer_name")\
                           .agg(F.avg("total_order_value").alias("avg_order_value"))
avg_order_value.show()

+-------------+---------------+
|customer_name|avg_order_value|
+-------------+---------------+
|      Charlie|          100.0|
|          Bob|           40.0|
|        Alice|           65.0|
+-------------+---------------+



In [33]:
# City-wise product revenue

city_product_revenue = full_data.withColumn("total_price", F.col("quantity") * F.col("price")) \
                                .groupBy("city", "product_name") \
                                .agg(F.sum("total_price").alias("total_revenue")) \
                                .orderBy("city", F.desc("total_revenue"))
city_product_revenue.show()

+---------+------------+-------------+
|     city|product_name|total_revenue|
+---------+------------+-------------+
|Bangalore|       Juice|         40.0|
|  Chennai|       Chips|         90.0|
|  Chennai|     Biscuit|         40.0|
|    Delhi|     Biscuit|        100.0|
+---------+------------+-------------+



In [34]:
# Total Orders and Revenue per Customer

customer_summary = full_data.withColumn("total_price", F.col("quantity")*F.col("price"))\
                            .groupBy("customer_name")\
                            .agg(F.countDistinct("order_id").alias("total_orders"),
                                 F.sum("total_price").alias("total_revenue"))\
                            .orderBy(F.desc("total_orders"))
customer_summary.show()

+-------------+------------+-------------+
|customer_name|total_orders|total_revenue|
+-------------+------------+-------------+
|        Alice|           2|        130.0|
|      Charlie|           1|        100.0|
|          Bob|           1|         40.0|
+-------------+------------+-------------+



In [35]:
# Quantity of each product bought by each customer

customer_product_pivot = full_data.groupBy("customer_name") \
                                  .pivot("product_name") \
                                  .agg(F.sum("quantity")) \
                                  .fillna(0)
customer_product_pivot.show()

+-------------+-------+-----+-----+
|customer_name|Biscuit|Chips|Juice|
+-------------+-------+-----+-----+
|      Charlie|    5.0|  0.0|  0.0|
|          Bob|    0.0|  0.0|  1.0|
|        Alice|    2.0|  3.0|  0.0|
+-------------+-------+-----+-----+

